In [1]:
import openeo
import json
from pathlib import Path

In [2]:
connection_cdse = openeo.connect("https://openeo.dataspace.copernicus.eu").authenticate_oidc()

Authenticated using refresh token.


In [3]:
!cwltool --pack ../../apex-force-openeo/cwl/force-tsa-workflow.cwl ../../apex-force-openeo/cwl/force-tsa.cwl ../../apex-force-openeo/cwl/force-tsa-parameter-schema.yaml > /tmp/force-tsa-packed.cwl

INFO /home/hannes/micromamba/envs/testing/bin/cwltool 3.1.20260315121657
INFO Resolved '../../apex-force-openeo/cwl/force-tsa-workflow.cwl' to 'file:///home/hannes/code/apex-force-openeo/cwl/force-tsa-workflow.cwl'


In [4]:
cwl = Path("/tmp/force-tsa-packed.cwl").read_text()

In [5]:
print(cwl)

{
    "$graph": [
        {
            "class": "Workflow",
            "requirements": [
                {
                    "types": [
                        {
                            "type": "enum",
                            "name": "#force-tsa-parameter-schema.yaml/STM_type",
                            "label": "Spectral Temporal Metrics to include",
                            "symbols": [
                                "#force-tsa-parameter-schema.yaml/STM_type/MIN",
                                "#force-tsa-parameter-schema.yaml/STM_type/MAX",
                                "#force-tsa-parameter-schema.yaml/STM_type/AVG",
                                "#force-tsa-parameter-schema.yaml/STM_type/STD",
                                "#force-tsa-parameter-schema.yaml/STM_type/RNG",
                                "#force-tsa-parameter-schema.yaml/STM_type/IQR",
                                "#force-tsa-parameter-schema.yaml/STM_type/SKW",
                        

In [6]:
force_datacube_url = "https://s3.waw4-1.cloudferro.com/calrissian/pvc-85434649-b4e1-4b9f-a44e-65533ed2e71f/j-26032413214843c79e-cal-cwl-760bd8d9/l2-ard/bologna-l2-ard.json"

In [7]:
force_pg = connection_cdse.datacube_from_process(
    "run_udf",
    data=None,
    udf=cwl,
    runtime="EOAP-CWL",
    context=dict(
        item_url=force_datacube_url,
        DATE_RANGE_START="2024-01-01",
        DATE_RANGE_END="2024-12-31",
    )
)
force_pg_json = force_pg.to_json()
print(force_pg_json)

{
  "process_graph": {
    "runudf1": {
      "process_id": "run_udf",
      "arguments": {
        "context": {
          "item_url": "https://s3.waw4-1.cloudferro.com/calrissian/pvc-85434649-b4e1-4b9f-a44e-65533ed2e71f/j-26032413214843c79e-cal-cwl-760bd8d9/l2-ard/bologna-l2-ard.json",
          "DATE_RANGE_START": "2024-01-01",
          "DATE_RANGE_END": "2024-12-31"
        },
        "data": null,
        "runtime": "EOAP-CWL",
        "udf": "{\n    \"$graph\": [\n        {\n            \"class\": \"Workflow\",\n            \"requirements\": [\n                {\n                    \"types\": [\n                        {\n                            \"type\": \"enum\",\n                            \"name\": \"#force-tsa-parameter-schema.yaml/STM_type\",\n                            \"label\": \"Spectral Temporal Metrics to include\",\n                            \"symbols\": [\n                                \"#force-tsa-parameter-schema.yaml/STM_type/MIN\",\n                  

In [8]:
force_job = connection_cdse.create_job(force_pg, title="FORCE TSA")
force_job.start_and_wait() # This still fails, because accessing the resulting datacube in FORCE format is not yet supported

0:00:00 Job 'j-26032416183845cc8da8f4bbfc026eed': send 'start'
0:00:17 Job 'j-26032416183845cc8da8f4bbfc026eed': created (progress 0%)
0:00:22 Job 'j-26032416183845cc8da8f4bbfc026eed': created (progress 0%)
0:00:29 Job 'j-26032416183845cc8da8f4bbfc026eed': created (progress 0%)
0:00:37 Job 'j-26032416183845cc8da8f4bbfc026eed': created (progress 0%)
0:00:47 Job 'j-26032416183845cc8da8f4bbfc026eed': running (progress N/A)
0:00:59 Job 'j-26032416183845cc8da8f4bbfc026eed': running (progress N/A)
0:01:15 Job 'j-26032416183845cc8da8f4bbfc026eed': running (progress N/A)
0:01:34 Job 'j-26032416183845cc8da8f4bbfc026eed': running (progress N/A)
0:01:58 Job 'j-26032416183845cc8da8f4bbfc026eed': running (progress N/A)
0:02:28 Job 'j-26032416183845cc8da8f4bbfc026eed': running (progress N/A)
0:03:05 Job 'j-26032416183845cc8da8f4bbfc026eed': running (progress N/A)
0:03:52 Job 'j-26032416183845cc8da8f4bbfc026eed': running (progress N/A)
0:04:51 Job 'j-26032416183845cc8da8f4bbfc026eed': running (progre

JobFailedException: Batch job 'j-26032416183845cc8da8f4bbfc026eed' didn't finish successfully. Status: error (after 0:06:51).

In [ ]:
force_logs = force_job.logs()

In [ ]:
filtered_logs = f'\n{"-"*80}\n\n”'.join([l.message for i, l in enumerate(force_logs) if "wrapper" in l.message or "wrapper" in force_logs[min(i+1, len(force_logs) - 1)].message])
print(filtered_logs)